# EZhire Model C - Jina
Trains jinaai/jina-embeddings-v2-base-en without chunking (8192-token limit).

In [ ]:
!pip install -q sentence-transformers datasets scikit-learn nltk PyMuPDF einops accelerate

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import nltk
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util, InputExample, losses
from sentence_transformers.evaluation import SentenceEvaluator
from torch.utils.data import DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

In [ ]:
ATS_MIN, ATS_MAX = 18.3, 90.7

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    m = re.search(r"\[SEP\]|\bSEP\b", text)
    if m:
        a = text[:m.start()]
        b = text[m.end():]
        return a.strip(), b.strip()
    mid = len(text) // 2
    return text[:mid].strip(), text[mid:].strip()

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def normalize_score(series):
    v = pd.to_numeric(series, errors='coerce').astype(float)
    return ((v - ATS_MIN) / (ATS_MAX - ATS_MIN)).clip(0, 1)

def denormalize_score(v):
    return np.asarray(v, float) * (ATS_MAX - ATS_MIN) + ATS_MIN

def build_df(src):
    splits = src['text'].apply(split_sep)
    ats = pd.to_numeric(src['ats_score'], errors='coerce').fillna(ATS_MIN)
    return pd.DataFrame({
        'resume_raw': splits.apply(lambda x: raw_text(x[0])),
        'jd_raw': splits.apply(lambda x: raw_text(x[1])),
        'resume_clean': splits.apply(lambda x: clean_text(x[0])),
        'jd_clean': splits.apply(lambda x: clean_text(x[1])),
        'original_label': src['original_label'].values,
        'ats_score_raw': ats.values,
        'ground_truth': normalize_score(ats).values
    }).dropna(subset=['resume_raw', 'jd_raw']).reset_index(drop=True)

ds = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_tr = build_df(df_train)
df_vl = build_df(df_val)
print(f'Train: {len(df_tr)} | Val: {len(df_vl)}')
print(f'GT range train: {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')
df_tr.head(2)

## Train Model C

In [ ]:
JINA_NAME = 'jinaai/jina-embeddings-v2-base-en'
JINA_PATH = os.path.join(MODEL_DIR, 'ezhire-jina')
EPOCHS_C = 4
BATCH_C = 2 if DEVICE == 'cuda' else 2
LR_C = 2e-5
WARMUP_C = 100
WD_C = 0.01
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)

print(f'Loading {JINA_NAME}...')
jina_model = SentenceTransformer(JINA_NAME, trust_remote_code=True, device=DEVICE)
jina_model.max_seq_length = 8192
print(f'Jina max_seq_length: {jina_model.max_seq_length}')

print('Building full-text training examples for Jina (no chunking)...')
jina_train_ex = [
    InputExample(texts=[row['resume_raw'], row['jd_raw']],
                 label=float(row['ground_truth']))
    for _, row in df_tr.iterrows()
]
print(f'Training examples: {len(jina_train_ex)} (one per row, full text)')

jina_loader = DataLoader(jina_train_ex, shuffle=True, batch_size=BATCH_C)
jina_loss = losses.CosineSimilarityLoss(jina_model)

class JinaDocEvaluator(SentenceEvaluator):
    def __init__(self, frame, save_path, name='jina_val'):
        self.frame = frame.reset_index(drop=True)
        self.save_path = save_path
        self.name = name
        self.best = -np.inf

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        resumes = self.frame['resume_raw'].tolist()
        jds = self.frame['jd_raw'].tolist()
        emb_r = model.encode(resumes, batch_size=4, show_progress_bar=False,
                             convert_to_tensor=True, normalize_embeddings=True)
        emb_j = model.encode(jds, batch_size=4, show_progress_bar=False,
                             convert_to_tensor=True, normalize_embeddings=True)
        sims = util.cos_sim(emb_r, emb_j).diagonal().cpu().numpy()
        yt = self.frame['ground_truth'].to_numpy(float)
        pe = float(pearsonr(sims, yt)[0]) if np.std(sims) > 0 and np.std(yt) > 0 else 0.0
        sp = float(spearmanr(sims, yt).correlation)
        pe = 0.0 if np.isnan(pe) else pe
        sp = 0.0 if np.isnan(sp) else sp
        mae = float(mean_absolute_error(yt, sims))
        print(f'[{self.name}] Pearson={pe:+.4f} Spearman={sp:+.4f} MAE={mae:.4f}')
        if pe > self.best:
            self.best = pe
            model.save(self.save_path)
            print(f'  Saved best Jina checkpoint -> {self.save_path}')
        return pe

val_sample_c = df_vl.sample(n=min(200, len(df_vl)), random_state=RANDOM_SEED)
jina_eval = JinaDocEvaluator(val_sample_c, save_path=JINA_PATH)

total_steps_c = len(jina_loader) * EPOCHS_C
warmup_c = min(WARMUP_C, max(1, total_steps_c // 10))
eval_steps_c = max(100, len(jina_loader) // 2)

print(f'Training: {EPOCHS_C} epochs | {len(jina_train_ex)} full-text pairs | batch {BATCH_C} | warmup {warmup_c}')

jina_model.fit(
    train_objectives=[(jina_loader, jina_loss)],
    evaluator=jina_eval,
    epochs=EPOCHS_C,
    warmup_steps=warmup_c,
    optimizer_params={'lr': LR_C},
    weight_decay=WD_C,
    evaluation_steps=eval_steps_c,
    output_path=None,
    save_best_model=False,
    show_progress_bar=True,
    use_amp=(DEVICE == 'cuda')
)

if not os.path.exists(os.path.join(JINA_PATH, 'modules.json')):
    jina_model.save(JINA_PATH)

jina_model = SentenceTransformer(JINA_PATH, trust_remote_code=True, device=DEVICE)
jina_model.max_seq_length = 8192
print('Model C (Jina) fine-tuning complete')

In [ ]:
print('Evaluating Model C (Jina) on full validation set (full text, no chunking)...')
jina_preds = []
for i, row in df_vl.iterrows():
    e1 = jina_model.encode(row['resume_raw'], convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    e2 = jina_model.encode(row['jd_raw'], convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    jina_preds.append(float(util.cos_sim(e1.unsqueeze(0), e2.unsqueeze(0))[0][0]))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')

df_vl['jina_score'] = [round(s * 100, 2) for s in jina_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['jina_score'].values
jina_pearson = float(pearsonr(yp / 100, yt)[0])
jina_spearman = float(spearmanr(yp / 100, yt).correlation)
jina_mae = float(mean_absolute_error(yt, yp / 100))
jina_rmse = float(np.sqrt(np.mean((denormalize_score(yt) - denormalize_score(yp / 100)) ** 2)))
jina_r2 = float(r2_score(yt, yp / 100))

print('')
print('Model C (Jina) Results:')
print(f'  Pearson  : {jina_pearson:+.4f}')
print(f'  Spearman : {jina_spearman:+.4f}')
print(f'  MAE_norm : {jina_mae:.4f}')
print(f'  RMSE_raw : {jina_rmse:.2f}')
print(f'  R2_raw   : {jina_r2:+.4f}')